### Libraries

In [1]:
import requests
import json
from bs4 import BeautifulSoup
import re
from datetime import datetime
import pandas as pd
import polars as pl

ModuleNotFoundError: No module named 'requests'

##### All APIs

In [ ]:
all_teams_url = "https://vlr.orlandomm.net/api/v1/teams?limit=all"
all_teams_response = requests.get(all_teams_url)
all_teams = all_teams_response.json()

In [ ]:
all_teams

In [ ]:
team_url = "https://vlr.orlandomm.net/api/v1/teams/2593"
team_response = requests.get(team_url)
team = team_response.json()

In [ ]:
events_url = "https://vlr.orlandomm.net/api/v1/matches?limit=100000"
events_response = requests.get(events_url)
events = events_response.json()

In [ ]:
results_url = 'https://vlr.orlandomm.net/api/v1/results'
results_response = requests.get(results_url)
results = results_response.json()
results

In [ ]:
team_url = "https://vlr.orlandomm.net/api/v1/results"
team_response = requests.get(team_url)
team = team_response.json()

In [ ]:
team['data'][20]
html = "https://www.vlr.gg/543173" 
site = requests.get(html)
match_html = BeautifulSoup(site.text, 'html.parser')

In [ ]:
match_html

In [ ]:
match_html.find_all('div', class_='match-header')[0].find('div', class_='match-header-vs').find_all('div', class_='wf-title-med')[1]#.text#.strip()

In [ ]:
text = match_html.find_all('div', class_='match-header-super')[0].find_all('a', class_='match-header-event')[0].text.strip()

# re.sub(r'\s+', ' ', text.strip())
text.split('\t')[0]

In [ ]:
timestamp = match_html.find('div', class_='match-header-date').text.split()#.get('data-utc-ts')
# datetime.strptime(timestamp, '%Y-%m-%d %H:%M:%S')
timestamp[-1]



### Map Dict

In [ ]:
map_dict = {
            'Ascent': 1,
            'Bind': 2,
            'Breeze': 3,
            'Fracture': 4,
            'Haven': 5,
            'Icebox': 6,
            'Lotus': 7,
            'Pearl': 8,
            'Split': 9,
            'Sunset': 10,
            'Abyss': 11,
            'Corrode': 12
        }
maps_played = []
map_info = match_html.find_all('div', class_ = 'vm-stats-gamesnav noselect')[0].text.strip()
map_info = re.sub(r'\s+', ' ', map_info).split()
for i in map_info:
    if i in map_dict:
        # map_info[i] = map_dict[i]
        maps_played.append(map_dict[i])
maps_played

In [ ]:
parser = VLRMatchParser(match_html)

parser.extract_players_dimension()

In [ ]:
class VLRMatchParser:
    """
    VLR Match Parser Class

    This class is used to parse the VLR match data and extract the dimensional/fact tables data.
    
    """

    def __init__(self, html_soup):
        self.soup = html_soup

    # Valorant Maps Dictionary
    def map_dict(self):
        map_dict = {
            'Ascent': 1,
            'Bind': 2,
            'Breeze': 3,
            'Fracture': 4,
            'Haven': 5,
            'Icebox': 6,
            'Lotus': 7,
            'Pearl': 8,
            'Split': 9,
            'Sunset': 10,
            'Abyss': 11,
            'Corrode': 12
        }
        return map_dict

    def extract_dimensional_data(self):
        """Extract all data organized by dimensional model"""
        return {
            # Dimension tables
            'matches': self.extract_match_dimension(),
            'maps': self.extract_maps_dimension(),
            'players': self.extract_players_dimension(), 
            'agents': self.extract_agents_dimension(),
            'teams': self.extract_teams_dimension(),
            'events': self.extract_events_dimension(),
            'economy': self.extract_economy_dimension(),
            
            # Fact tables
            'match_performance': self.extract_player_stats(),
            'match_overall_stats': self.extract_match_overall_stats(),
            'match_economy': self.extract_economy_data()
        }
        
    def extract_match_dimension(self):
        """Extract comprehensive match information"""
        match_info = {}
        
        # Extract match header information
        match_header = self.soup.find('div', class_='match-header')
        if match_header:
            # Extract teams
            teams = self.soup.find_all('div', class_='match-header-vs')
            
            match_info['team1'] = teams[0].find_all('div', class_='wf-title-med')[0].text.strip() 
                # if teams[0].find_all('div', class_='wf-title-med')[0] else 'Unknown'
            match_info['team2'] = teams[0].find_all('div', class_='wf-title-med')[1].text.strip()
                # if teams[0].find_all('div', class_='wf-title-med')[1] else 'Unknown'

            match_info['event'] = match_html.find_all('div', class_='match-header-super')[0].find_all('a', class_='match-header-event')[0].text.strip()
            
        
        # Extract event info
        event_elem = self.soup.find('div', class_='match-header-super')
        if event_elem:
            match_info['event'] = event_elem.find_all('a', class_='match-header-event')[0].text.strip().split('\t')[0]
            match_info['event_id'] = re.findall(r'event/(\d+)', event_elem.find_all('a', class_='match-header-event')[0].get('href'))[0]
        
        # Extract date
        date_elem = self.soup.find('div', class_='moment-tz-convert').get('data-utc-ts')
        if date_elem:
            datetime_element = datetime.strptime(date_elem, '%Y-%m-%d %H:%M:%S')
            match_info['match_date'] = datetime_element.date()

        
        # Extract score
        score_elem = self.soup.find('div', class_='match-header-vs-score')
        if score_elem:
            score_int = score_elem.text.strip().split()
            match_info['score'] = [str(item) for item in score_int if item.isdigit()]

        # Extract VLR link and image source
        team_link_1 = match_html.find('a', class_ = 'match-header-link wf-link-hover mod-1')
        team_link_2 = match_html.find('a', class_ = 'match-header-link wf-link-hover mod-2')

        if team_link_1 and team_link_2:
            team_href_1 = team_link_1.get('href')
            team_href_2 = team_link_2.get('href')

            team_logo_1 = team_link_1.find('img')
            team_logo_2 = team_link_2.find('img')
            
            logo_src_1 = team_logo_1.get('src')
            logo_src_2 = team_logo_2.get('src')
            
        #     match_info['team1_logo'] = logo_src_1
        #     match_info['team2_logo'] = logo_src_2
            
        #     match_info['team1_href'] = team_href_1
        #     match_info['team2_href'] = team_href_2
            
            
        match_note = match_html.find('div', class_ = 'match-header-note').text.strip()
        match_info['match_note'] = match_note
        match_info['team_1_id'] = re.findall(r'team/(\d+)', team_href_1)[0]
        match_info['team_2_id'] = re.findall(r'team/(\d+)', team_href_2)[0]

        match_info['match_id'] = re.findall(r'(\d+)', match_html.find_all('div', class_ = 'vm-stats')[0].get('data-url'))[0]

        # Map Information
        map_info = match_html.find_all('div', class_ = 'vm-stats-gamesnav noselect')[0].text.strip()
        map_info = re.sub(r'\s+', ' ', map_info).split()
        map_dict = self.map_dict()
        maps_played = []
        for map_id in map_info:
            if map_id in map_dict:
                maps_played.append(map_dict[map_id])
        match_info['maps_played'] = maps_played

        match_patch = match_html.find('div', class_='match-header-date').text.split()
        match_info['match_patch'] = match_patch[-1]

        
        return match_info
    
    

    
    # def extract_maps_dimension(self):
    #     """Extract unique maps for dimension table"""
    #     maps = set()
    #     map_containers = self.soup.find_all('div', class_='vm-stats-game')
        
    #     for container in map_containers:
    #         if container.get('data-game-id') == 'all':
    #             continue
                
    #         map_elem = container.find('div', class_='map')
    #         if map_elem:
    #             map_name_span = map_elem.find('span')
    #             if map_name_span:
    #                 # Clean the map name by removing tabs, newlines, and extra whitespace
    #                 map_name = re.sub(r'\s+', ' ', map_name_span.get_text().strip())
    #                 # Extract only the map name (before any pick/ban info)
    #                 map_name = map_name.split('PICK')[0].split('BAN')[0].strip()
    #                 if map_name:
    #                     maps.add(map_name)
        
    #     return [{'map_name': map_name} for map_name in maps]
    
    def extract_players_dimension(self):
        """Extract unique players for dimension table"""
        players = {}
        
        # Get player data from the overall stats view
        all_container = self.soup.find('div', class_='vm-stats-game', attrs={'data-game-id': 'all'})
        if all_container:
            tables = all_container.find_all('table', class_='wf-table-inset')
            
            for table in tables:
                rows = table.find('tbody').find_all('tr') if table.find('tbody') else []
                
                for row in rows:
                    player_cell = row.find('td', class_='mod-player')
                    if player_cell:
                        player_link = player_cell.find('a')
                        if player_link:
                            # Clean player name by removing tabs, newlines, and extra whitespace
                            player_name = re.sub(r'\s+', ' ', player_link.get_text().strip())
                            player_data = {
                                'player_name': player_name,
                                'player_url': player_link.get('href', ''),
                            }
                        
                        # Get country
                        flag_elem = player_cell.find('i', class_='flag')
                        if flag_elem:
                            player_data['country'] = flag_elem.get('title', '')
                            flag_classes = flag_elem.get('class', [])
                            for cls in flag_classes:
                                if cls.startswith('mod-'):
                                    player_data['country_code'] = cls.replace('mod-', '').upper()
                                    break
                        
                                                        # Get team
                        team_elem = player_cell.find('div', class_='ge-text-light')
                        if team_elem:
                            # Clean team name by removing tabs, newlines, and extra whitespace
                            team_name = re.sub(r'\s+', ' ', team_elem.get_text().strip())
                            player_data['current_team'] = team_name
                        
                        players[player_name] = player_data
        
        return list(players.values())
    
    def extract_agents_dimension(self):
        """Extract unique agents for dimension table"""
        agents = set()
        
        all_container = self.soup.find('div', class_='vm-stats-game', attrs={'data-game-id': 'all'})
        if all_container:
            agent_imgs = all_container.find_all('img', src=lambda x: x and 'agents' in x)
            
            for img in agent_imgs:
                agent_name = img.get('alt', '').title()
                if agent_name:
                    agents.add((agent_name, img.get('src', '')))
        
        return [{'agent_name': name, 'agent_image_url': url} for name, url in agents]
    
    def extract_teams_dimension(self):
        """Extract team information for dimension table"""
        match_info = self.extract_match_info()
        teams = []
        
        # Extract from match header
        if 'team1' in match_info and 'team2' in match_info:
            teams.append({
                'team_name': match_info['team1'],
                'team_logo': match_info.get('team1_logo', ''),
                'team_url': match_info.get('team1_href', '')
            })
            teams.append({
                'team_name': match_info['team2'], 
                'team_logo': match_info.get('team2_logo', ''),
                'team_url': match_info.get('team2_href', '')
            })
        
        return teams
    
    def extract_events_dimension(self):
        """Extract event information for dimension table"""
        match_info = self.extract_match_info()
        
        return [{
            'event_name': match_info.get('event', ''),
            'match_note': match_info.get('match_note', ''),
            'match_date': match_info.get('match_date', None)
        }]
    
    def extract_match_overall_stats(self):
        """Extract overall match statistics for fact table"""
        match_stats = []
        
        # Get overall stats from the 'all' view
        all_container = self.soup.find('div', class_='vm-stats-game', attrs={'data-game-id': 'all'})
        if all_container:
            # Team-level aggregated stats could be extracted here
            # This would include team totals, averages, etc.
            
            teams_data = self.extract_team_performance_stats(all_container)
            match_stats.extend(teams_data)
        
        return match_stats
    
    def extract_team_performance_stats(self, container):
        """Extract team-level performance statistics"""
        team_stats = []
        
        # Extract team stats from the aggregated player data
        tables = container.find_all('table', class_='wf-table-inset')
        
        for table_idx, table in enumerate(tables):
            team_data = {
                'team_number': table_idx + 1,
                'game_id': 'all',
                'total_kills': 0,
                'total_deaths': 0,
                'total_assists': 0,
                'avg_acs': 0,
                'avg_adr': 0
            }
            
            rows = table.find('tbody').find_all('tr') if table.find('tbody') else []
            player_count = 0
            
            for row in rows:
                player_cell = row.find('td', class_='mod-player')
                if player_cell and player_cell.find('a'):  # Valid player row
                    player_count += 1
                    
                    # Sum up team stats from individual players
                    stat_cells = row.find_all('td', class_='mod-stat')
                    if len(stat_cells) >= 5:
                        # Extract kills, deaths, assists
                        kills_elem = stat_cells[2].find('span', class_='mod-both')
                        deaths_elem = stat_cells[3].find('span', class_='mod-both') 
                        assists_elem = stat_cells[4].find('span', class_='mod-both')
                        acs_elem = stat_cells[1].find('span', class_='mod-both')
                        
                        if kills_elem:
                            team_data['total_kills'] += self.parse_stat_value(kills_elem.text.strip(), 'kills') or 0
                        if deaths_elem:
                            team_data['total_deaths'] += self.parse_stat_value(deaths_elem.text.strip(), 'deaths') or 0
                        if assists_elem:
                            team_data['total_assists'] += self.parse_stat_value(assists_elem.text.strip(), 'assists') or 0
                        if acs_elem:
                            acs_val = self.parse_stat_value(acs_elem.text.strip(), 'acs') or 0
                            team_data['avg_acs'] += acs_val
            
            # Calculate averages
            if player_count > 0:
                team_data['avg_acs'] = team_data['avg_acs'] / player_count
                team_data['player_count'] = player_count
            
            team_stats.append(team_data)
        
        return team_stats


In [ ]:
# Test the updated VLRMatchParser with dimensional model
parser = VLRMatchParser(match_html)

# Test individual methods
print("=== MATCH INFO ===")
match_info = parser.extract_match_info()
print(f"Teams: {match_info.get('team1')} vs {match_info.get('team2')}")
print(f"Event: {match_info.get('event')}")
print(f"Date: {match_info.get('match_date')}")
print(f"Score: {match_info.get('score')}")

print("\n=== MAPS DATA ===")
maps_data = parser.extract_map_data()
print(f"Number of maps: {len(maps_data)}")
for i, map_data in enumerate(maps_data):
    print(f"Map {i+1}: {map_data.get('map_name')} - {map_data.get('team1_score')}:{map_data.get('team2_score')}")

print("\n=== PLAYER STATS (SAMPLE) ===")
player_stats = parser.extract_player_stats()
print(f"Total player records: {len(player_stats)}")
if player_stats:
    sample_player = player_stats[0]
    print(f"Sample player: {sample_player.get('player_name')} ({sample_player.get('team_tag')})")
    print(f"Agent: {sample_player.get('agent')}")
    print(f"Stats: K:{sample_player.get('kills')} D:{sample_player.get('deaths')} A:{sample_player.get('assists')}")

print("\n=== DIMENSIONAL DATA OVERVIEW ===")
dimensional_data = parser.extract_dimensional_data()
for key, value in dimensional_data.items():
    if isinstance(value, list):
        print(f"{key}: {len(value)} records")
    else:
        print(f"{key}: {type(value).__name__}")


In [ ]:
# Detailed example of the dimensional model structure
print("=== DIMENSIONAL MODEL STRUCTURE ===")
print("\n1. DIMENSION TABLES:")

# Teams dimension
teams = dimensional_data['teams']
print(f"\nTeams ({len(teams)} records):")
for team in teams:
    print(f"  - {team}")

# Maps dimension  
maps = dimensional_data['maps']
print(f"\nMaps ({len(maps)} records):")
for map_info in maps:
    print(f"  - {map_info}")

# Players dimension (first 3)
players = dimensional_data['players']
print(f"\nPlayers ({len(players)} records, showing first 3):")
for player in players[:3]:
    print(f"  - {player}")

# Agents dimension
agents = dimensional_data['agents']
print(f"\nAgents ({len(agents)} records):")
for agent in agents:
    print(f"  - {agent}")

print("\n2. FACT TABLES:")

# Match Performance (player stats)
performance = dimensional_data['match_performance']
print(f"\nMatch Performance ({len(performance)} records, showing first record):")
if performance:
    print(f"  Sample: {performance[0]}")

# Match Overall Stats
overall_stats = dimensional_data['match_overall_stats']
print(f"\nMatch Overall Stats ({len(overall_stats)} records):")
for stat in overall_stats:
    print(f"  - {stat}")

print("\n=== DATA MODEL COMPLIANCE ===")
print("✓ Dimension Tables: Matches, Events, Players, Agents, Maps, Teams")
print("✓ Fact Tables: Match Performance, Match Overall Stats, Match Economy")
print("✓ Proper separation of dimensional and transactional data")
print("✓ Referential integrity through game_id and player names")


In [ ]:
# Test the cleaned data extraction
parser = VLRMatchParser(match_html)
dimensional_data = parser.extract_dimensional_data()

print("=== CLEANED MAPS DIMENSION ===")
maps = dimensional_data['maps']
print(f"Maps ({len(maps)} records):")
for map_info in maps:
    print(f"  - {map_info}")

print("\n=== CLEANED PLAYERS DIMENSION ===")
players = dimensional_data['players']
print(f"Players ({len(players)} records, showing first 3):")
for player in players[:3]:
    print(f"  - {player}")

print("\n=== VERIFICATION ===")
# Check for any remaining tabs or newlines
for map_info in maps:
    map_name = map_info['map_name']
    if '\t' in map_name or '\n' in map_name:
        print(f"❌ Map still has whitespace issues: {repr(map_name)}")
    else:
        print(f"✓ Map name clean: {map_name}")

print()
for player in players[:3]:
    player_name = player['player_name']
    if '\t' in player_name or '\n' in player_name:
        print(f"❌ Player still has whitespace issues: {repr(player_name)}")
    else:
        print(f"✓ Player name clean: {player_name}")
        
    if 'current_team' in player:
        team_name = player['current_team']
        if '\t' in team_name or '\n' in team_name:
            print(f"❌ Team still has whitespace issues: {repr(team_name)}")
        else:
            print(f"✓ Team name clean: {team_name}")


In [ ]:
parser.extract_match_overall_stats()

In [ ]:
class SupabaseDataInserter:
    def __init__(self, supabase_client):
        self.supabase = supabase_client
    
    def insert_or_get_team(self, team_name, team_tag=None, region=None, country=None):
        """Insert team or get existing team ID"""
        try:
            # Check if team exists
            result = self.supabase.table('dim_teams').select('team_id').eq('team_name', team_name).execute()
            
            if result.data:
                return result.data[0]['team_id']
            
            # Insert new team
            team_data = {
                'team_name': team_name,
                'team_tag': team_tag,
                'region': region,
                'country': country
            }
            
            result = self.supabase.table('dim_teams').insert(team_data).execute()
            return result.data[0]['team_id']
            
        except Exception as e:
            print(f"Error inserting team {team_name}: {e}")
            return None
    
    def insert_or_get_player(self, player_name, team_id, player_tag=None, real_name=None, country=None, role=None):
        """Insert player or get existing player ID"""
        try:
            # Check if player exists
            result = self.supabase.table('dim_players').select('player_id').eq('player_name', player_name).execute()
            
            if result.data:
                return result.data[0]['player_id']
            
            # Insert new player
            player_data = {
                'player_name': player_name,
                'player_tag': player_tag,
                'real_name': real_name,
                'country': country,
                'team_id': team_id,
                'role': role
            }
            
            result = self.supabase.table('dim_players').insert(player_data).execute()
            return result.data[0]['player_id']
            
        except Exception as e:
            print(f"Error inserting player {player_name}: {e}")
            return None
    
    def insert_or_get_event(self, event_name, event_type=None, start_date=None, end_date=None):
        """Insert event or get existing event ID"""
        try:
            # Check if event exists
            result = self.supabase.table('dim_events').select('event_id').eq('event_name', event_name).execute()
            
            if result.data:
                return result.data[0]['event_id']
            
            # Insert new event
            event_data = {
                'event_name': event_name,
                'event_type': event_type,
                'start_date': start_date,
                'end_date': end_date
            }
            
            result = self.supabase.table('dim_events').insert(event_data).execute()
            return result.data[0]['event_id']
            
        except Exception as e:
            print(f"Error inserting event {event_name}: {e}")
            return None
    
    def get_agent_id(self, agent_name):
        """Get agent ID by name"""
        try:
            result = self.supabase.table('dim_agents').select('agent_id').eq('agent_name', agent_name).execute()
            if result.data:
                return result.data[0]['agent_id']
            return None
        except Exception as e:
            print(f"Error getting agent {agent_name}: {e}")
            return None
    
    def get_map_id(self, map_name):
        """Get map ID by name"""
        try:
            result = self.supabase.table('dim_maps').select('map_id').eq('map_name', map_name).execute()
            if result.data:
                return result.data[0]['map_id']
            return None
        except Exception as e:
            print(f"Error getting map {map_name}: {e}")
            return None
    
    def insert_match(self, vlr_match_id, event_id, team1_id, team2_id, match_date, match_format='BO3', match_status='completed', winner_team_id=None, final_score=None):
        """Insert match data"""
        try:
            match_data = {
                'vlr_match_id': vlr_match_id,
                'event_id': event_id,
                'team1_id': team1_id,
                'team2_id': team2_id,
                'match_date': match_date.isoformat() if match_date else None,
                'match_format': match_format,
                'match_status': match_status,
                'winner_team_id': winner_team_id,
                'final_score': final_score
            }
            
            result = self.supabase.table('dim_matches').insert(match_data).execute()
            return result.data[0]['match_id']
            
        except Exception as e:
            print(f"Error inserting match: {e}")
            return None
    
    def insert_match_performance(self, match_id, player_id, team_id, map_id, agent_id, stats):
        """Insert player performance data"""
        try:
            performance_data = {
                'match_id': match_id,
                'player_id': player_id,
                'team_id': team_id,
                'map_id': map_id,
                'agent_id': agent_id,
                'kills': stats.get('kills', 0),
                'deaths': stats.get('deaths', 0),
                'assists': stats.get('assists', 0),
                'acs': stats.get('acs', 0),
                'adr': stats.get('adr', 0),
                'hs_percentage': stats.get('hs_percentage', 0),
                'first_kills': stats.get('first_kills', 0),
                'first_deaths': stats.get('first_deaths', 0),
                'fkfd_diff': stats.get('first_kills', 0) - stats.get('first_deaths', 0),
                'rating': stats.get('rating', 0)
            }
            
            result = self.supabase.table('fact_match_performance').insert(performance_data).execute()
            return result.data[0]['performance_id']
            
        except Exception as e:
            print(f"Error inserting performance data: {e}")
            return None
    
    def insert_match_overall_stats(self, match_id, team_id, map_id, team_stats):
        """Insert team overall stats for a map"""
        try:
            overall_data = {
                'match_id': match_id,
                'team_id': team_id,
                'map_id': map_id,
                'rounds_won': team_stats.get('rounds_won', 0),
                'rounds_lost': team_stats.get('rounds_lost', 0),
                'total_kills': team_stats.get('total_kills', 0),
                'total_deaths': team_stats.get('total_deaths', 0),
                'total_assists': team_stats.get('total_assists', 0),
                'total_acs': team_stats.get('total_acs', 0),
                'total_adr': team_stats.get('total_adr', 0),
                'first_kills': team_stats.get('first_kills', 0),
                'first_deaths': team_stats.get('first_deaths', 0),
                'clutches_won': team_stats.get('clutches_won', 0),
                'clutches_attempted': team_stats.get('clutches_attempted', 0)
            }
            
            result = self.supabase.table('fact_match_overall_stats').insert(overall_data).execute()
            return result.data[0]['stat_id']
            
        except Exception as e:
            print(f"Error inserting overall stats: {e}")
            return None


In [ ]:
def process_match_data(html_soup, vlr_match_id, supabase_inserter):
    """Main function to process match data and insert into database"""
    
    # Initialize parser
    parser = VLRMatchParser(html_soup)
    
    # Extract basic match info
    match_info = parser.extract_match_info()
    print(f"Processing match: {match_info}")
    
    # Insert/get teams
    team1_id = supabase_inserter.insert_or_get_team(match_info.get('team1', 'Unknown Team 1'))
    team2_id = supabase_inserter.insert_or_get_team(match_info.get('team2', 'Unknown Team 2'))
    
    # Insert/get event
    event_id = supabase_inserter.insert_or_get_event(match_info.get('event', 'Unknown Event'))
    
    # Determine winner based on score
    winner_team_id = None
    score = match_info.get('score', '')
    if ':' in score:
        scores = score.split(':')
        if len(scores) == 2:
            try:
                score1, score2 = int(scores[0].strip()), int(scores[1].strip())
                if score1 > score2:
                    winner_team_id = team1_id
                elif score2 > score1:
                    winner_team_id = team2_id
            except ValueError:
                pass
    
    # Insert match
    match_id = supabase_inserter.insert_match(
        vlr_match_id=vlr_match_id,
        event_id=event_id,
        team1_id=team1_id,
        team2_id=team2_id,
        match_date=match_info.get('match_date'),
        final_score=score,
        winner_team_id=winner_team_id
    )\n    
    if not match_id:
        print("Failed to insert match data")
        return
    
    # Extract and process map data
    maps_data = parser.extract_map_data()
    
    for map_data in maps_data:
        map_name = map_data.get('map_name', 'Unknown')
        map_id = supabase_inserter.get_map_id(map_name)
        
        if not map_id:
            print(f"Map {map_name} not found in database")
            continue
        
        # Process players for this map
        team_stats = {team1_id: {'kills': 0, 'deaths': 0, 'assists': 0, 'acs': 0, 'adr': 0}, 
                     team2_id: {'kills': 0, 'deaths': 0, 'assists': 0, 'acs': 0, 'adr': 0}}
        
        for player_data in map_data.get('players', []):
            player_name = player_data.get('player_name')
            team_name = player_data.get('team')
            
            if not player_name:
                continue
            
            # Determine team ID
            current_team_id = team1_id if team_name == match_info.get('team1') else team2_id
            
            # Insert/get player
            player_id = supabase_inserter.insert_or_get_player(player_name, current_team_id)
            
            # Get agent ID
            agent_name = player_data.get('agent')
            agent_id = supabase_inserter.get_agent_id(agent_name) if agent_name else None
            
            # Insert performance data
            supabase_inserter.insert_match_performance(
                match_id=match_id,
                player_id=player_id,
                team_id=current_team_id,
                map_id=map_id,
                agent_id=agent_id,
                stats=player_data
            )
            
            # Aggregate team stats
            if current_team_id in team_stats:
                team_stats[current_team_id]['kills'] += player_data.get('kills', 0)
                team_stats[current_team_id]['deaths'] += player_data.get('deaths', 0)
                team_stats[current_team_id]['assists'] += player_data.get('assists', 0)
                team_stats[current_team_id]['acs'] += player_data.get('acs', 0)
                team_stats[current_team_id]['adr'] += player_data.get('adr', 0)
        
        # Insert team overall stats for this map
        for team_id, stats in team_stats.items():
            if stats['kills'] > 0:  # Only insert if team has data
                # Calculate averages
                stats['total_acs'] = stats['acs'] / 5 if stats['acs'] > 0 else 0
                stats['total_adr'] = stats['adr'] / 5 if stats['adr'] > 0 else 0
                
                supabase_inserter.insert_match_overall_stats(
                    match_id=match_id,
                    team_id=team_id,
                    map_id=map_id,
                    team_stats=stats
                )
    
    print(f"Successfully processed match {vlr_match_id}")
    return match_id


In [ ]:
# Test the parser with your current match data
parser = VLRMatchParser(match_html)

# Extract basic match information
match_info = parser.extract_match_info()
print("Match Info:")
print(json.dumps(match_info, indent=2, default=str))

# Extract map data
maps_data = parser.extract_map_data()
print(f"\nFound {len(maps_data)} maps")

for i, map_data in enumerate(maps_data):
    print(f"\nMap {i+1}: {map_data.get('map_name', 'Unknown')}")
    print(f"Score: {map_data.get('score', 'N/A')}")
    print(f"Players found: {len(map_data.get('players', []))}")
    
    # Show first few players as example
    for j, player in enumerate(map_data.get('players', [])[:3]):
        print(f"  Player {j+1}: {player.get('player_name', 'Unknown')} - {player.get('agent', 'Unknown')} - {player.get('kills', 0)}K/{player.get('deaths', 0)}D/{player.get('assists', 0)}A")


In [ ]:
# Example usage (uncomment when you have Supabase credentials)
"""
# Initialize Supabase client with your credentials
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# Initialize data inserter
inserter = SupabaseDataInserter(supabase)

# Process the current match
match_id = process_match_data(
    html_soup=match_html,
    vlr_match_id="543173",  # Extract from URL
    supabase_inserter=inserter
)

print(f"Processed match with ID: {match_id}")
"""

# For now, let's just demonstrate the parsing without database insertion
print("To use the database functionality:")
print("1. Set up your Supabase project")
print("2. Run the SQL schema file to create tables")
print("3. Update SUPABASE_URL and SUPABASE_KEY variables")
print("4. Uncomment the example usage code above")


### RIBs.GG

In [4]:
BASE_URL = "https://be-prod.rib.gg/v1"

def get_all_events():
    """Fetch all events using skip/take pagination"""
    all_events = []
    skip = 0
    take = 100
    
    while True:
        url = f"{BASE_URL}/events?skip={skip}&take={take}"
        response = requests.get(url)
        data = response.json()
        
        events = data.get('data', [])
        all_events.extend(events)
        
        total = data['meta']['total']
        print(f"Fetched {len(all_events)} / {total} events...")
        
        if len(all_events) >= total or len(events) == 0:
            break
        
        skip += take
    
    return all_events

In [8]:
events_df = get_all_events()

Fetched 100 / 6413 events...
Fetched 200 / 6413 events...
Fetched 300 / 6413 events...
Fetched 400 / 6413 events...
Fetched 500 / 6413 events...
Fetched 600 / 6413 events...
Fetched 700 / 6413 events...
Fetched 800 / 6413 events...
Fetched 900 / 6413 events...
Fetched 1000 / 6413 events...
Fetched 1100 / 6413 events...
Fetched 1200 / 6413 events...
Fetched 1300 / 6413 events...
Fetched 1400 / 6413 events...
Fetched 1500 / 6413 events...
Fetched 1600 / 6413 events...
Fetched 1700 / 6413 events...
Fetched 1800 / 6413 events...
Fetched 1900 / 6413 events...
Fetched 2000 / 6413 events...
Fetched 2100 / 6413 events...
Fetched 2200 / 6413 events...
Fetched 2300 / 6413 events...
Fetched 2400 / 6413 events...
Fetched 2500 / 6413 events...
Fetched 2600 / 6413 events...
Fetched 2700 / 6413 events...
Fetched 2800 / 6413 events...
Fetched 2900 / 6413 events...
Fetched 3000 / 6413 events...
Fetched 3100 / 6413 events...
Fetched 3200 / 6413 events...
Fetched 3300 / 6413 events...
Fetched 3400 / 6413

In [10]:
df = pl.DataFrame(events_df, infer_schema_length=None)
df.sort("startDate")

id,name,shortName,description,formatMd,eventsMd,logoUrl,regionId,countryId,startDate,endDate,prizePool,prizePoolCurrency,url,imageUrl,livestreamLink,winnerStageCount,loserStageCount,live,rank,pmtJson,parent,parentId,childLabel,keywords,slug,seriesCount,importance,type,liquipediaSlug,vctRegions,divisions,t3Subdivision,region,country
i64,str,str,str,str,str,str,i64,i64,str,str,i64,str,str,str,str,i64,i64,bool,i64,null,bool,i64,str,list[str],str,i64,i64,str,str,list[str],list[str],str,struct[4],struct[6]
79,"""G2 Esports Invitational""",null,"""The G2 Esports Invitational wa…",null,null,null,1,null,"""2020-06-19T17:27:00.000Z""","""2020-06-21T17:27:00.000Z""",15000,"""EUR""","""https://g2esports.com/valorant…","""https://images.dexerto.com/upl…",null,null,null,false,3,null,false,null,null,null,null,12,0,"""event""",null,null,null,null,"{1,""Europe"",""eu"",""europe""}",null
4,"""T1 x Nerd Street Gamers Showdo…",null,null,null,null,null,2,null,"""2020-06-25T18:53:00.000Z""","""2020-06-27T18:53:00.000Z""",50000,"""USD""","""https://liquipedia.net/valoran…","""https://i.imgur.com/JxNhrVG.pn…",null,null,null,false,3,null,false,null,null,[],null,1,0,"""event""",null,null,null,null,"{2,""North America"",""na"",""north-america""}",null
29,"""AfreecaTV Asia Invitational Su…",null,"""AfreecaTV Asia Invitational Su…",null,null,null,3,null,"""2020-07-23T06:16:00.000Z""","""2020-08-13T06:16:00.000Z""",5000,"""USD""","""https://twitter.com/aftvgame_j…","""https://i.imgur.com/VsoojNP.jp…",null,null,null,false,3,null,false,null,null,null,null,0,0,"""event""",null,null,null,null,"{3,""Asia-Pacific"",""ap"",""asia-pacific""}",null
109,"""GLL Orbital Strike Cup""",null,"""We are very excited to announc…",null,null,null,4,null,"""2020-07-29T14:32:00.000Z""","""2020-08-05T14:32:00.000Z""",5000,"""BRL""","""https://play.gll.gg/valorant/t…","""https://i.imgur.com/LRcZFT5.jp…",null,null,null,false,3,null,false,null,null,null,null,0,0,"""event""",null,null,null,null,"{4,""Latin America"",""latam"",""latin-america""}",null
31,"""Allied Esports Strafe Series""",null,"""Welcome to Allied Esports Nort…",null,null,null,2,null,"""2020-08-04T07:28:00.000Z""","""2020-09-01T07:28:00.000Z""",2500,"""USD""","""https://smash.gg/tournament/al…","""https://i.imgur.com/XZfMhOb.jp…",null,null,null,false,3,null,true,null,null,null,null,0,0,"""event""",null,null,null,null,"{2,""North America"",""na"",""north-america""}",null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
7272,"""Challengers League 2026: Brazi…","""Playoffs""",null,null,null,null,null,30,"""2026-07-03T20:00:00.000Z""","""2026-07-12T20:00:00.000Z""",null,null,"""https://liquipedia.net/valoran…",null,null,null,null,false,null,null,false,7227,"""Playoffs""",[],"""valorant-challengers-2026-braz…",0,0,"""bracket""","""VCL/2026/Brazil/Stage_2/playof…","[""AMERICAS""]","[""VCL""]",null,null,"{30,""brazil"",""Brazil"",""BR"",""BRA"",76}"
7247,"""Challengers League 2026: Brazi…","""VCL Brazil 2026: Stage 2 Releg…","""Part of the Valorant Challenge…",null,null,"""https://i.imgur.com/zXIHjKz.we…",null,30,"""2026-07-14T00:00:00.000Z""","""2026-07-16T00:00:00.000Z""",0,null,"""https://liquipedia.net/valoran…","""https://liquipedia.net/commons…",null,null,null,false,null,null,true,6848,null,[],"""valorant-challengers-2026-braz…",0,0,"""event""","""VCL/2026/Brazil/Stage_2/Relega…","[""AMERICAS""]","[""VCL""]",null,null,"{30,""brazil"",""Brazil"",""BR"",""BRA"",76}"
7248,"""Challengers League 2026: Brazi…","""Relegation""",null,null,null,null,null,30,"""2026-07-14T20:00:00.000Z""","""2026-07-16T20:00:00.000Z""",null,null,"""https://liquipedia.net/valoran…",null,null,null,null,false,null,null,false,7247,"""Relegation""",[],"""valorant-challengers-2026-braz…",0,0,"""bracket""","""VCL/2026/Brazil/Stage_2/Relega…","[""AMERICAS""]","[""VCL""]",null,null,"{30,""brazil"",""Brazil"",""BR"",""BRA"",76}"


In [11]:
import time
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://be-prod.rib.gg/v1"

def get_session_with_retries():
    """Create a requests session with automatic retries"""
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

def get_all_series(team1_id=9060, team2_id=5675):
    """Fetch head-to-head series between two teams"""
    session = get_session_with_retries()
    url = f"{BASE_URL}/series/head-to-head?team1Id={team1_id}&team2Id={team2_id}"
    response = session.get(url, timeout=30)
    response.raise_for_status()
    data = response.json()
    return data

data = get_all_series(team1_id=9060, team2_id=5675)
data

{'team1': {'id': 9060,
  'name': 'G2 Esports',
  'shortName': 'G2',
  'description': '',
  'websiteUrl': 'https://g2esports.com/',
  'logoUrl': 'https://i.imgur.com/v3lSzoH.webp',
  'liquipediaSlug': 'G2_Esports',
  'twitterUrl': 'https://twitter.com/G2esports',
  'twitchUrl': 'https://www.twitch.tv/g2esports',
  'vlrUrl': 'https://www.vlr.gg/team/11058/g2-esports',
  'youtubeUrl': 'https://www.youtube.com/FollowGamers2',
  'foundedDate': '2022-12-12T00:00:00.000Z',
  'countryId': 226,
  'regionId': 2,
  'rank': 1544,
  'regionRank': 616,
  'aliases': None,
  'vctRegion': 'AMERICAS',
  'division': 'VCT',
  'grid_team_id': '96',
  'players': [{'id': 369,
    'ign': 'BABYBAY',
    'firstName': 'Andrej',
    'lastName': 'Francisty',
    'bio': 'Andrej "babybay" Francisty (born June 23, 1995) is an American player who currently plays for FaZe Clan. He is a retired professional Overwatch player.',
    'countryId': 226,
    'instagramUrl': 'https://www.instagram.com/king_babybay/',
    'liqu

In [ ]:
import time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://be-prod.rib.gg/v1"

def get_session_with_retries():
    """Create a requests session with automatic retries"""
    session = requests.Session()
    retry = Retry(
        total=5,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

def get_all_series():
    """Fetch all series with retry logic and pagination"""
    session = get_session_with_retries()
    all_series = []
    skip = 0
    take = 100
    
    while True:
        try:
            url = f"{BASE_URL}/series?skip={skip}&take={take}"
            response = session.get(url, timeout=30)
            response.raise_for_status()
            data = response.json()
        except Exception as e:
            print(f"Error at skip={skip}: {e}")
            print("Waiting 10s before retrying...")
            time.sleep(10)
            continue
        
        series = data.get('data', [])
        if len(series) == 0:
            break
            
        all_series.extend(series)
        
        total = data['meta']['total']
        print(f"Fetched {len(all_series)} / {total} series...")
        
        if len(all_series) >= total:
            break
        
        skip += take
        time.sleep(0.2)
    
    return all_series

series = get_all_series()
sdf = pd.DataFrame(series)

Fetched 100 / 98754 series...
Fetched 200 / 98754 series...
Fetched 300 / 98754 series...
Fetched 400 / 98754 series...
Fetched 500 / 98754 series...
Fetched 600 / 98754 series...
Fetched 700 / 98754 series...
Fetched 800 / 98754 series...
Fetched 900 / 98754 series...
Fetched 1000 / 98754 series...
Fetched 1100 / 98754 series...
Fetched 1200 / 98754 series...
Fetched 1300 / 98754 series...
Fetched 1400 / 98754 series...
Fetched 1500 / 98754 series...
Fetched 1600 / 98754 series...
Fetched 1700 / 98754 series...
Fetched 1800 / 98754 series...
Fetched 1900 / 98754 series...
Fetched 2000 / 98754 series...
Fetched 2100 / 98754 series...
Fetched 2200 / 98754 series...
Fetched 2300 / 98754 series...
Fetched 2400 / 98754 series...
Fetched 2500 / 98754 series...
Fetched 2600 / 98754 series...
Fetched 2700 / 98754 series...
Fetched 2800 / 98754 series...
Fetched 2900 / 98754 series...
Fetched 3000 / 98754 series...
Fetched 3100 / 98754 series...
Fetched 3200 / 98754 series...
Fetched 3300 / 98

In [1]:
import time
import threading
import polars as pl
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from concurrent.futures import ThreadPoolExecutor, as_completed

BASE_URL = "https://be-prod.rib.gg/v1"
TAKE = 100
MAX_WORKERS = 10

_lock = threading.Lock()
_completed = 0

def _make_session():
    session = requests.Session()
    retry = Retry(total=5, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])
    adapter = HTTPAdapter(max_retries=retry)
    session.mount("https://", adapter)
    session.mount("http://", adapter)
    return session

def _fetch_page(skip, total_pages):
    global _completed
    worker = threading.current_thread().name
    session = _make_session()
    resp = session.get(f"{BASE_URL}/series?skip={skip}&take={TAKE}", timeout=30)
    resp.raise_for_status()
    data = resp.json()
    count = len(data.get("data", []))
    with _lock:
        _completed += 1
        print(f"[{worker}] skip={skip} — got {count} series (page {_completed}/{total_pages})")
    return data

def get_all_series_parallel():
    global _completed
    _completed = 0

    first = _fetch_page(0, 1)
    total = first["meta"]["total"]
    all_series = first.get("data", [])
    total_pages = (total + TAKE - 1) // TAKE
    _completed = 1
    print(f"\nTotal series: {total} ({total_pages} pages) — fetching with {MAX_WORKERS} workers...\n")

    remaining_skips = list(range(TAKE, total, TAKE))

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(_fetch_page, s, total_pages): s for s in remaining_skips}
        for fut in as_completed(futures):
            try:
                page = fut.result()
                all_series.extend(page.get("data", []))
            except Exception as e:
                print(f"Error at skip={futures[fut]}: {e}")

    print(f"\nFetched {len(all_series)} / {total} series")
    return all_series

t0 = time.perf_counter()
series = get_all_series_parallel()
sdf = pl.DataFrame(series, infer_schema_length=None, strict=False)
print(f"Done in {time.perf_counter() - t0:.1f}s — {sdf.shape}")
sdf.head()

[MainThread] skip=0 — got 100 series (page 1/1)

Total series: 98754 (988 pages) — fetching with 10 workers...

[ThreadPoolExecutor-0_3] skip=400 — got 100 series (page 2/988)
[ThreadPoolExecutor-0_0] skip=100 — got 100 series (page 3/988)
[ThreadPoolExecutor-0_5] skip=600 — got 100 series (page 4/988)
[ThreadPoolExecutor-0_2] skip=300 — got 100 series (page 5/988)
[ThreadPoolExecutor-0_7] skip=800 — got 100 series (page 6/988)
[ThreadPoolExecutor-0_6] skip=700 — got 100 series (page 7/988)
[ThreadPoolExecutor-0_4] skip=500 — got 100 series (page 8/988)
[ThreadPoolExecutor-0_9] skip=1000 — got 100 series (page 9/988)
[ThreadPoolExecutor-0_1] skip=200 — got 100 series (page 10/988)
[ThreadPoolExecutor-0_8] skip=900 — got 100 series (page 11/988)
[ThreadPoolExecutor-0_3] skip=1100 — got 100 series (page 12/988)
[ThreadPoolExecutor-0_0] skip=1200 — got 100 series (page 13/988)
[ThreadPoolExecutor-0_5] skip=1300 — got 100 series (page 14/988)
[ThreadPoolExecutor-0_2] skip=1400 — got 100 se

TypeError: unexpected value while building Series of type Float64; found value of type Int64: 0

Hint: Try setting `strict=False` to allow passing data with mixed types.

In [2]:
sdf = pl.DataFrame(series, infer_schema_length=None, strict=False)

In [5]:
sdf.shape

(98754, 35)

In [8]:
# import json

# matches = sdf[sdf['id'] == 99925]['matches'].values[0]
# print(json.dumps(matches, indent=2))

sdf.write_parquet("All_Series_List.parquet")

In [ ]:
url = "https://be-prod.rib.gg/v1/matches/227686/details"
response = requests.get(url)
data = response.json()
data.keys()

In [ ]:
data

In [ ]:
import json

matches = sdf[sdf['id'] == 99925]['matches'].values[0]
print(json.dumps(matches, indent=2))

In [2]:
import os
import psycopg2 as psy
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

# os.path.join("")
load_dotenv()

url: str = os.environ.get("DATABASE_URL")

# connection = psy.connect(url)

engine = create_engine(url)


with engine.connect() as conn:
        result = conn.execute(text("SELECT 100+10"))
        latest_season = result.scalar()

print(latest_season)

110


In [ ]:

DIM
get all events
get all matches
get all teams
get all players
get all agents - one time
get all maps - one time
get all weapons - one time

FACT
get all overall match stats, player stats incl.
get all player events
get all player stats
get all player economies
get all player locations
get all player kill locations